In [6]:
import rasterio
import geopandas as gpd
from rasterio.features import geometry_mask
import numpy as np
import os

import tqdm

from shapely.geometry import box





In [12]:
def process_rasters(raster_files, target_value, polygons_gdf, output_dir):
    """
    Process rasters to generate binary masks where polygons overlap with pixels of target_value.
    
    Parameters:
    raster_files (list): List of file paths to the input raster files.
    target_value (int or float): The target pixel value to consider.
    polygons_gdf (geopandas.GeoDataFrame): GeoDataFrame of polygons.
    output_dir (str): Directory to save the output raster files.
    """
    for i, raster_file in enumerate(raster_files):
        # Read the raster
        with rasterio.open(raster_file) as src:
            raster_data = src.read(1)
            raster_transform = src.transform
            raster_crs = src.crs
            raster_shape = raster_data.shape
            
            # Subset polygons that intersect with the raster bounds
            raster_bounds =  box(*src.bounds) 
            intersecting_polygons = polygons_gdf[polygons_gdf.intersects(raster_bounds)]
            
            if intersecting_polygons.empty:
                print(f"No intersecting polygons for raster {raster_file}")
                continue
            
            # Create a mask for the target_value
            target_mask = (raster_data == target_value)
            
            # Create a mask for the intersecting polygons
            poly_mask = geometry_mask(intersecting_polygons.geometry, transform=raster_transform,
                                      invert=True, out_shape=raster_shape)
            
            # Combine the masks
            combined_mask = np.logical_and(target_mask, poly_mask).astype(np.uint8)
            
            # Define the output raster path
            output_raster_path = os.path.join(output_dir, os.path.basename(raster_file))
            
            # Write the output raster
            with rasterio.open(
                output_raster_path.replace('ESA_WorldCover_10m_', ''),
                'w',
                driver='GTiff',
                height=raster_shape[0],
                width=raster_shape[1],
                count=1,
                dtype=np.uint8,
                crs=raster_crs,
                transform=raster_transform,
                compress='lzw'
            ) as dst:
                dst.write(combined_mask, 1)
                
            print(f"Processed raster saved at {output_raster_path}")



In [13]:
# Example usage
import glob
raster_files = glob.glob('../data/input_datasets/worldcover/*.tif')  # List of raster file paths
target_value = 30 
polygons_gdf = gpd.read_file('../data/input_datasets/Deforestation_Drivers_Data_archive_from_Camilo/COL_pasture.shp')  # Load your polygons GeoDataFrame
output_dir = '../data/input_datasets/colombia_pasture'  # Output directory for processed rasters
from pathlib import Path
Path(output_dir).mkdir(parents=True, exist_ok=True)


In [14]:

import sys
sys.path.append('../')
from utils import get_tiles_as_gdf

worldcover_gdf = get_tiles_as_gdf('../data/input_datasets/worldcover/')
intersecting_worldcover_gdf = worldcover_gdf.loc[worldcover_gdf.intersects(box(*polygons_gdf.total_bounds))]

In [15]:
process_rasters(intersecting_worldcover_gdf['filename'], target_value, polygons_gdf, output_dir)

Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N00W069_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N00W072_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N00W075_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N00W078_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N00W081_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N03W069_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N03W072_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N03W075_Map.tif
Processed raster saved at ../data/input_datasets/colombia_pasture/ESA_WorldCover_10m_2021_v200_N03W078_Map.tif
N